# CSCI 4150 — Evaluating AI Systems: Mini-Lab
**Tuesday, November 17, 2026 · Rensselaer Polytechnic Institute**

**Learning outcomes:** identify misleading evaluation results, select task-appropriate metrics, detect leakage, test robustness, and write a defensible evaluation claim.

**Environment:** Google Colab or local Jupyter. No APIs, credentials, downloads, GPU, or model access required. Recommended in class: **Exercises 1 and 3 (~15 minutes)**; Exercises 2 and 4 for extended practice or team preparation.

**Important:** All data/results in this notebook are **synthetic teaching examples**, not empirical claims about a real AI system.

Open in Colab: *File → Upload notebook*. Run cells sequentially.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
print("Setup complete.")

ModuleNotFoundError: No module named 'pandas'

## Exercise 1 — The accuracy trap (5 minutes)
**Scenario:** A support-ticket classifier predicts whether tickets are urgent. Only 10 of 100 are urgent. Model A predicts *not urgent* for every example. Model B finds most urgent cases but creates some false alarms.

**Before running:** Predict which model has higher accuracy; which is safer if missing urgent tickets is costly?

In [ ]:
# 90 non-urgent, 10 urgent tickets
actual = np.array([0] * 90 + [1] * 10)
model_a = np.zeros(100, dtype=int)
# Model B: 18 false positives, 8 true positives
model_b = np.array([1] * 18 + [0] * 72 + [1] * 8 + [0] * 2)

def metrics(y_true, y_pred):
    return dict(accuracy=accuracy_score(y_true,y_pred),
                precision=precision_score(y_true,y_pred,zero_division=0),
                recall=recall_score(y_true,y_pred,zero_division=0),
                f1=f1_score(y_true,y_pred,zero_division=0))

results = pd.DataFrame({'All-negative baseline':metrics(actual,model_a),
                        'Model B':metrics(actual,model_b)}).T
print(results.round(3).to_string())
print('\nModel B confusion matrix (rows: actual 0/1; columns: predicted 0/1):')
print(confusion_matrix(actual,model_b))

**Your turn:**
1. What happens to positive-class recall when a model predicts only the majority class?
2. For high-risk missed urgent tickets, would you optimize precision, recall, or a cost-sensitive measure? Explain the tradeoff.
3. What additional baseline would you compare against before claiming model improvement?

**Check:** The all-negative baseline achieves high accuracy but zero urgent-class recall. Model B trades some accuracy for higher recall.

## Exercise 2 — Data leakage: random rows versus unseen groups (7 minutes)
**Scenario:** Each customer has multiple tickets. A model can memorize which customers have historically received an urgent label. A random *row* split may put the same customer into training and test sets. That can inflate measured performance when the real application must generalize to **new** customers.

We create a **deliberately artificial** dataset where each customer's label is random but stays constant across that customer's rows. Customer identity is therefore a shortcut, not a meaningful rule.

In [ ]:
rng=np.random.default_rng(4150)
num_groups, repeats=60, 8
group_labels=rng.integers(0,2,num_groups)
ids=np.repeat(np.arange(num_groups),repeats)
y=np.repeat(group_labels,repeats)
X=[{'customer':f'C{g:03d}'} for g in ids]

def train_and_measure(train_idx,test_idx):
    vec=DictVectorizer(sparse=True)
    X_train=vec.fit_transform([X[i] for i in train_idx])
    X_test=vec.transform([X[i] for i in test_idx])
    clf=LogisticRegression(max_iter=1000)
    clf.fit(X_train,y[train_idx])
    return accuracy_score(y[test_idx],clf.predict(X_test)), len(set(ids[train_idx]) & set(ids[test_idx]))

indices=np.arange(len(y))
row_train,row_test=train_test_split(indices,test_size=0.25,random_state=12,stratify=y)
gss=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=12)
group_train,group_test=next(gss.split(indices,y,groups=ids))
row_accuracy,row_overlap=train_and_measure(row_train,row_test)
group_accuracy,group_overlap=train_and_measure(group_train,group_test)
print(pd.DataFrame([
    {'split':'Random ROW split','accuracy':row_accuracy,'shared customers':row_overlap},
    {'split':'CUSTOMER-group split','accuracy':group_accuracy,'shared customers':group_overlap}
]).to_string(index=False))

**Your turn:** Explain why the first test can look strong even though the system has learned nothing useful about new customers. List one analogous leakage risk for your project (e.g., near-duplicate documents, the same user, same product, or temporal information).

**Interpretation note:** Here the group split models a *new-customer* deployment. If your real task is prediction for returning customers, a different time-ordered evaluation may be appropriate. Choose a split that matches the claim.

## Exercise 3 — Robustness: meaning-preserving perturbations (5 minutes)
**Scenario:** The original classifier detects a ticket as urgent only when it contains the exact lowercase word `urgent`. Try uppercase, punctuation, and a typo. Not all typos are meaning-preserving; the chosen test cases should be reviewed.

In [ ]:
inputs = [
  ('Urgent: payment outage', 1),
  ('URGENT: payment outage', 1),
  ('urgent!!! payment outage', 1),
  ('payment outage: urgently need help', 1),
  ('Routine account question', 0),
  ('not urgent - informational only', 0),
]

def brittle_rule(message):
    return int('urgent' in message.split())

rows=[]
for message, expected in inputs:
    predicted=brittle_rule(message)
    rows.append({'input':message,'expected':expected,'predicted':predicted,'pass':predicted==expected})
df=pd.DataFrame(rows)
print(df.to_string(index=False))
print('Pass rate:', round(df['pass'].mean(),3))

**Important semantic caveat:** The sentence *“not urgent”* should be negative. A simplistic keyword method might flag it as urgent; normalization alone cannot solve negation or context.

**Task:** Implement an improved rule below. It should handle case and punctuation but avoid interpreting *“not urgent”* as urgent. Then run the cell to test it. This is an exercise in failure-driven revision—not a claim that a keyword rule is ready for deployment.

In [ ]:
import re

def improved_rule(message):
    # TODO: adapt your rule. This starting point intentionally remains simple.
    tokens = re.findall(r"[a-z]+", message.lower())
    if 'urgent' in tokens and 'not' not in tokens:
        return 1
    return 0

comparison=pd.DataFrame([
    {'input':m,'expected':e,'brittle':brittle_rule(m),'improved':improved_rule(m)}
    for m,e in inputs
])
comparison['brittle_correct']=comparison.expected==comparison.brittle
comparison['improved_correct']=comparison.expected==comparison.improved
print(comparison.to_string(index=False))
print('Original accuracy:',comparison.brittle_correct.mean())
print('Revised accuracy:',comparison.improved_correct.mean())
print('Failures remaining:',comparison.loc[~comparison.improved_correct,'input'].tolist())

**Discussion:** Is an improved aggregate score sufficient? What happens on the new case *“I am not sure if this is urgent”*? Why should a held-out robustness set be used after changing the rule? Record one failure category and one mitigation or restriction.

## Exercise 4 — Design an evaluation plan for YOUR project (8–10 minutes)
Complete this short template. No code or special libraries are required. Focus on a **specific behavioral claim** (classification, extraction, retrieval, generation, agent workflow, or structured audit). The suggested 10–20 starter cases are preparation for Friday, **not** a claim of statistical sufficiency.

In [ ]:
evaluation_plan = {
  'system_claim': 'Our system can ... for ...',
  'intended_population': 'Inputs/users/context ...',
  'primary_metric_or_rubric': 'Metric AND why it matches the claim ...',
  'baseline': 'Simple rule / majority class / existing process ...',
  'ground_truth_procedure': 'Who labels? Which rubric? How handle disagreement? ...',
  'split_strategy': 'Held-out by time, user, source, group, or other ...',
  'representative_cases': 'Types of ordinary cases ...',
  'challenge_cases': 'Ambiguity, missing values, edge cases, adversarial ...',
  'robustness_test': 'Transformation + expected invariant behavior ...',
  'failure_slices': 'Conditions whose errors should be compared ...',
  'readiness_boundary': 'What must the system not do alone? ...'
}
for k,v in evaluation_plan.items():
    print(f'{k.replace("_"," ").title()}: {v}')

### Evidence log (copy and fill)
This mirrors the evaluation summaries required for A3 and the curated evidence record in A4.

In [ ]:
from IPython.display import display
evidence_log=pd.DataFrame(columns=[
    'test_id','input_or_condition','expected_behavior','observed_result',
    'metric_or_verdict','failure_category','design_implication'])
display(evidence_log)
# To create your first entry:
# evidence_log.loc[len(evidence_log)] = ['T01','...', '...', '...', 'pass/fail', '...', '...']

## Sample interpretation / instructor discussion key

- **Exercise 1:** An all-negative classifier has 90% accuracy and 0% positive recall. Model B intentionally sacrifices some accuracy to detect urgent cases. Evaluate false-negative/false-positive costs and an operational baseline.
- **Exercise 2:** A row split allows customer IDs to appear on both sides; strong scores may reflect memorization. Group separation removes the shortcut for unseen customers. A different split might be justified for returning-customer deployment.
- **Exercise 3:** Case normalization fixes some format sensitivity but does not establish contextual understanding. Maintain challenge examples such as negation and phrases like “urgently.”
- **Exercise 4:** A strong plan ties a **claim** to a **population**, **reference labels**, **metric**, **holdout**, **robustness tests**, and **design decisions**.

**For Friday:** Each team should bring one claim, initial cases and expected outcomes, a primary metric or rubric, an intended split, and at least one challenge slice. Run evaluations on the project itself and record failures. A3 requires an evaluation plan/test set, failure analysis, red-team evidence, and readiness judgment. A4 carries the final evidence and limitations into the Project Dossier.

**Course milestones:** A3 due **Tuesday, December 1, 2026**; A4 due **Friday, December 11, 2026**. Refer to course postings for exact submission times.